In [1]:
# ============================================================
# Step 7 — Per-tumor NNMF (k=10 × 6 tumors)
# Cell 1: load Filbin_step6.h5ad, confirm anchors, resolve the
#         per-tumor-centered layer name (do NOT assume).
# ============================================================
import scanpy as sc
import numpy as np
import pandas as pd

adata = sc.read_h5ad("../data/processed/Filbin_step6.h5ad")

# --- shape anchor (D9: 2,456 from Step 6 forward) ---
print("shape:", adata.shape)                      # expect (2456, 8494)
assert adata.shape == (2456, 8494), "Step 6 shape anchor broke"

# --- Step 7 input keys from Step 6 ---
assert "is_malignant" in adata.obs, "missing is_malignant"
assert "step6_ambiguous" in adata.obs, "missing step6_ambiguous"
n_mal = int(adata.obs["is_malignant"].sum())
n_nm  = int((~adata.obs["is_malignant"]).sum())
n_amb = int(adata.obs["step6_ambiguous"].sum())
print(f"is_malignant: {n_mal} malignant / {n_nm} NM")   # expect 2329 / 127
print(f"step6_ambiguous (kept in, flagged): {n_amb}")    # expect 77
assert n_mal == 2329 and n_nm == 127, "is_malignant anchor drifted"
assert n_amb == 77, "step6_ambiguous anchor drifted"

# --- malignant cells per tumor (the NNMF input populations) ---
mal = adata.obs["is_malignant"].to_numpy()
print("\nmalignant cells per tumor:")
print(adata.obs.loc[mal, "patient"].value_counts())
# expect MUV5 704, BCH869 489, BCH836 441, MUV10 286, BCH1126 263, MUV1 146

# --- RESOLVE the per-tumor-centered layer name (no guessing) ---
# Step 7 needs per-tumor Er (NOT centered_global, which was Step-6 CNV only).
print("\nlayers present:", sorted(adata.layers.keys()))
candidates = [k for k in adata.layers if "center" in k.lower()]
print("centered-layer candidates:", candidates)

# Per-tumor centered layer: its per-tumor column means must be ~0 (≈1e-6 float32),
# while a GLOBAL-centered layer has per-tumor means that are NOT ~0.
def per_tumor_mean_max(layer_name):
    M = adata.layers[layer_name]
    M = M.toarray() if hasattr(M, "toarray") else np.asarray(M)
    worst = 0.0
    for t in adata.obs["patient"].unique():
        m = (adata.obs["patient"] == t).to_numpy()
        worst = max(worst, np.abs(M[m].mean(axis=0)).max())
    return worst

for k in candidates:
    print(f"  {k}: max |per-tumor gene mean| = {per_tumor_mean_max(k):.2e}")
# The PER-TUMOR layer is the one with max |per-tumor mean| ~1e-6.
# centered_global will show a much larger value here.

shape: (2456, 8494)
is_malignant: 2329 malignant / 127 NM
step6_ambiguous (kept in, flagged): 77

malignant cells per tumor:
MUV5       704
BCH869     489
BCH836     441
MUV10      286
BCH1126    263
MUV1       146
Name: patient, dtype: int64

layers present: ['centered_global', 'centered_within_tumor', 'tpm']
centered-layer candidates: ['centered_global', 'centered_within_tumor']
  centered_global: max |per-tumor gene mean| = 4.47e+00
  centered_within_tumor: max |per-tumor gene mean| = 2.28e-02


In [2]:
# ============================================================
# Step 7 — Cell 2: per-tumor NNMF (k=10 × 6 tumors)
# Substrate: centered_within_tumor (Er), malignant cells only,
#            negatives clipped to 0 (Filbin non-negativity step).
# Config:    solver='mu', init='random', beta_loss='frobenius',
#            random_state=42  (faithful Matlab nnmf analogue, one run/tumor)
# ============================================================
from sklearn.decomposition import NMF
import numpy as np
import pandas as pd

K = 10                      # Filbin anchor: 10 factors per tumor
TOP_N = 30                  # Filbin anchor: top 30 genes per factor → signature
SEED = 42
TUMORS = ["MUV1", "MUV5", "MUV10", "BCH836", "BCH869", "BCH1126"]

genes = np.asarray(adata.var_names)
Er_full = adata.layers["centered_within_tumor"]
Er_full = Er_full.toarray() if hasattr(Er_full, "toarray") else np.asarray(Er_full)

mal_mask = adata.obs["is_malignant"].to_numpy()
patient  = adata.obs["patient"].to_numpy()

# Containers
W_by_tumor = {}            # gene × factor loadings (W)
H_by_tumor = {}            # factor × cell activations (H), cells = that tumor's malignant
cells_by_tumor = {}        # cell names, row order of H
signatures = {}            # {"MUV1_f0": [30 gene symbols], ...}
recon_err = {}

for t in TUMORS:
    cell_mask = mal_mask & (patient == t)
    cell_names = adata.obs_names[cell_mask].to_numpy()
    n_cells = cell_mask.sum()

    # Er for this tumor's malignant cells, negatives → 0 (non-negativity for NNMF)
    M = Er_full[cell_mask, :]               # (n_cells, 8494)
    M = np.clip(M, 0.0, None)               # Filbin: "set negative values to zero"

    # Drop all-zero genes for THIS tumor (a gene never positive here carries no signal;
    # leaving it in just adds zero rows to W). We keep a full-length W by reindexing back.
    nz_gene = M.sum(axis=0) > 0
    M_nz = M[:, nz_gene]

    model = NMF(
        n_components=K,
        init="random",
        solver="mu",
        beta_loss="frobenius",
        max_iter=1000,
        random_state=SEED,
    )
    H_cells = model.fit_transform(M_nz)     # (n_cells, K)  — cell activations
    W_genes_nz = model.components_.T         # (n_genes_nz, K) — gene loadings

    # Reindex W back to full 8,494-gene space (zeros for dropped genes)
    W_full = np.zeros((adata.n_vars, K), dtype=np.float64)
    W_full[np.where(nz_gene)[0], :] = W_genes_nz

    W_by_tumor[t] = W_full
    H_by_tumor[t] = H_cells
    cells_by_tumor[t] = cell_names
    recon_err[t] = float(model.reconstruction_err_)

    # Top-30-gene signature per factor (ranked by W loading)
    for f in range(K):
        top_idx = np.argsort(W_full[:, f])[::-1][:TOP_N]
        signatures[f"{t}_f{f}"] = genes[top_idx].tolist()

    print(f"{t:8s} n_mal={n_cells:4d}  genes_used={nz_gene.sum():5d}  "
          f"recon_err={recon_err[t]:.1f}")

print(f"\ntotal candidate programs: {len(signatures)} (expect 60)")
assert len(signatures) == 60

# Quick peek: top-10 genes of factor 0 for each tumor (sanity, not a decision)
print("\nfactor 0 top-10 per tumor (raw, unordered across tumors):")
for t in TUMORS:
    print(f"  {t:8s}: {signatures[f'{t}_f0'][:10]}")

MUV1     n_mal= 146  genes_used= 8469  recon_err=1318.6
MUV5     n_mal= 704  genes_used= 8487  recon_err=2441.8
MUV10    n_mal= 286  genes_used= 8456  recon_err=1838.4
BCH836   n_mal= 441  genes_used= 8491  recon_err=1935.3
BCH869   n_mal= 489  genes_used= 8474  recon_err=2305.1
BCH1126  n_mal= 263  genes_used= 8480  recon_err=1548.7

total candidate programs: 60 (expect 60)

factor 0 top-10 per tumor (raw, unordered across tumors):
  MUV1    : ['SPARCL1', 'APOE', 'AGT', 'AQP4', 'EDNRB', 'CLU', 'HOPX', 'MLC1', 'GJA1', 'LGALS3']
  MUV5    : ['SRSF9', 'PPP4C', 'GLTSCR2', 'C20orf24', 'AZGP1', 'CCDC85B', 'STUB1', 'HES6', 'NME4', 'SURF1']
  MUV10   : ['HERC2P4', 'LOC100190986', 'UGDH-AS1', 'OPHN1', 'LOC643406', 'ARHGEF26-AS1', 'LPAL2', 'ABCC9', 'LOC646214', 'TMEM212']
  BCH836  : ['GPR17', 'SIRT2', 'RGR', 'BCAS1', 'TNS3', 'GPC3', 'PTGDS', 'S100A10', 'SCRG1', 'OSR1']
  BCH869  : ['AGT', 'CLU', 'AQP4', 'MLC1', 'HLA-C', 'GJA1', 'HLA-B', 'SPARCL1', 'ATP1A2', 'APOE']
  BCH1126 : ['CA10', 'LPPR1'

In [4]:
# Inspect Table S5 structure before writing the loader
import pandas as pd

S5_PATH = "/home/kian/Downloads/aao4750_filbin_sm_tables5.xlsx"

xl = pd.ExcelFile(S5_PATH)
print("sheet names:", xl.sheet_names)
print()
for sh in xl.sheet_names:
    df = pd.read_excel(S5_PATH, sheet_name=sh, header=None, nrows=6)
    print(f"=== sheet '{sh}' — shape {pd.read_excel(S5_PATH, sheet_name=sh, header=None).shape} ===")
    print(df.to_string(max_cols=12))
    print()

sheet names: ['Sheet1']

=== sheet 'Sheet1' — shape (53, 5) ===
                                                       0      1     2           3             4
0  Table S5: Gene expression signatures in H3K27M-Glioma    NaN   NaN         NaN           NaN
1                                                    NaN    NaN   NaN         NaN           NaN
2                                              Cellcycle     OC    AC  OPC-shared  OPC-variable
3                                                  UBE2T  BCAS1  AQP4      PDGFRA        PDGFRA
4                                                  HMGB2   PLP1   CLU        MEST         ITM2C
5                                                   TYMS  PTGDS   AGT       CCND1          SCG3



/home/kian/anaconda3/envs/dmg-py/lib/python3.11/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning: Unknown extension is not supported and will be removed
  for idx, row in parser.parse():


In [5]:
# ============================================================
# Step 7 — Cell 3a: load Filbin Table S5 program gene lists
# Layout: title row 0, blank row 1, headers row 2, genes row 3+
# ============================================================
import pandas as pd

S5_PATH = "/home/kian/Downloads/aao4750_filbin_sm_tables5.xlsx"

s5 = pd.read_excel(S5_PATH, sheet_name="Sheet1", header=2)  # row 2 = column names
print("columns as read:", list(s5.columns))

# Build {program: set(genes)}, dropping NaN padding, normalizing header names
# to the keys used elsewhere in the pipeline (underscore form).
rename = {
    "Cellcycle": "Cellcycle",
    "OC": "OC",
    "AC": "AC",
    "OPC-shared": "OPC_shared",
    "OPC-variable": "OPC_variable",
}
prog_lists = {}
for col, key in rename.items():
    genes_col = s5[col].dropna().astype(str).str.strip()
    genes_col = genes_col[genes_col != ""].tolist()
    prog_lists[key] = set(genes_col)

# Report sizes + coverage against our 8,494-gene matrix
working = set(adata.var_names)
print("\nprogram list sizes (and how many are in our 8,494-gene matrix):")
for k, v in prog_lists.items():
    print(f"  {k:13s}: {len(v):3d} genes   in-matrix {len(v & working):3d}")

# Sanity vs handoff anchors: Cellcycle/OC/AC=50, OPC_shared=19, OPC_variable=50
expected = {"Cellcycle": 50, "OC": 50, "AC": 50, "OPC_shared": 19, "OPC_variable": 50}
for k, n in expected.items():
    got = len(prog_lists[k])
    flag = "" if got == n else f"  <-- expected {n}"
    if got != n:
        print(f"  NOTE {k}: {got} genes{flag}")

columns as read: ['Cellcycle', 'OC', 'AC', 'OPC-shared', 'OPC-variable']

program list sizes (and how many are in our 8,494-gene matrix):
  Cellcycle    :  50 genes   in-matrix  50
  OC           :  50 genes   in-matrix  50
  AC           :  50 genes   in-matrix  50
  OPC_shared   :  19 genes   in-matrix  19
  OPC_variable :  50 genes   in-matrix  50


/home/kian/anaconda3/envs/dmg-py/lib/python3.11/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning: Unknown extension is not supported and will be removed
  for idx, row in parser.parse():


In [6]:
# ============================================================
# Step 7 — Cell 3: concordance check — overlap each of the 60
# factors' top-30 against Filbin Table S5 program lists.
# Validation only; no decisions, no freeze.
# prog_lists is live from Cell 3a (5 base programs).
# ============================================================
import pandas as pd
import numpy as np

TUMORS = ["MUV1", "MUV5", "MUV10", "BCH836", "BCH869", "BCH1126"]
prog_names = list(prog_lists.keys())   # Cellcycle, OC, AC, OPC_shared, OPC_variable

rows = []
for t in TUMORS:
    for f in range(10):
        sig = set(signatures[f"{t}_f{f}"])
        row = {"factor": f"{t}_f{f}"}
        for p in prog_names:
            row[p] = len(sig & prog_lists[p])
        best = max(prog_names, key=lambda p: row[p])
        row["best_match"] = best if row[best] > 0 else "—"
        row["best_overlap"] = row[best]
        rows.append(row)

overlap = pd.DataFrame(rows).set_index("factor")
pd.set_option("display.max_rows", 70)
print(overlap[prog_names + ["best_match", "best_overlap"]].to_string())

print("\nbest-match counts across the 60 factors:")
print(overlap["best_match"].value_counts())

print("\nprograms strongly captured per tumor (best_overlap >= 5):")
strong = overlap[overlap["best_overlap"] >= 5]
for t in TUMORS:
    hits = sorted(strong.loc[[i for i in strong.index if i.startswith(t)], "best_match"].unique())
    print(f"  {t:8s}: {hits}")

            Cellcycle  OC  AC  OPC_shared  OPC_variable    best_match  best_overlap
factor                                                                             
MUV1_f0             0   0  16           0             1            AC            16
MUV1_f1             0   0   3           3             0            AC             3
MUV1_f2             1   4   0           1             6  OPC_variable             6
MUV1_f3             0   0   0           0             0             —             0
MUV1_f4             0   0   0           0             5  OPC_variable             5
MUV1_f5            22   0   0           0             0     Cellcycle            22
MUV1_f6             0   8   1           0             0            OC             8
MUV1_f7             1   0   0           1             0     Cellcycle             1
MUV1_f8             0   0   5           0             0            AC             5
MUV1_f9             0   0   0           0             4  OPC_variable       

In [7]:
# Full table, no truncation + per-tumor coverage matrix
import pandas as pd
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 200)
print(overlap[prog_names + ["best_match", "best_overlap"]].to_string())

# Coverage matrix: for each tumor, the MAX overlap achieved on each program
# (did the tumor capture this program at all, and how strongly?)
print("\n=== per-tumor MAX overlap per program ===")
cov = (overlap.assign(tumor=[i.rsplit("_f", 1)[0] for i in overlap.index])
              .groupby("tumor")[prog_names].max()
              .reindex(["MUV1","MUV5","MUV10","BCH836","BCH869","BCH1126"]))
print(cov.to_string())

# Which factors are "unmatched" (best_overlap < 3) — tumor-specific / technical
print("\n=== unmatched factors (best_overlap < 3) ===")
print(overlap[overlap["best_overlap"] < 3].index.tolist())

            Cellcycle  OC  AC  OPC_shared  OPC_variable    best_match  best_overlap
factor                                                                             
MUV1_f0             0   0  16           0             1            AC            16
MUV1_f1             0   0   3           3             0            AC             3
MUV1_f2             1   4   0           1             6  OPC_variable             6
MUV1_f3             0   0   0           0             0             —             0
MUV1_f4             0   0   0           0             5  OPC_variable             5
MUV1_f5            22   0   0           0             0     Cellcycle            22
MUV1_f6             0   8   1           0             0            OC             8
MUV1_f7             1   0   0           1             0     Cellcycle             1
MUV1_f8             0   0   5           0             0            AC             5
MUV1_f9             0   0   0           0             4  OPC_variable       

In [8]:
# ============================================================
# Step 7 — Cell 3b: gene-level inspection of MUV10's factors
# and the split cell-cycle factors. Diagnostic only.
# ============================================================

# MUV10's UNMATCHED factors (best_overlap < 3): f0,f1,f2,f4,f5,f7,f9
print("=== MUV10 unmatched factors — top 15 genes ===")
for f in [0, 1, 2, 4, 5, 7, 9]:
    print(f"MUV10_f{f}:", signatures[f"MUV10_f{f}"][:15])

# MUV10's MATCHED factors for contrast: f3 (CC 18), f6 (CC 20), f8 (OPCvar 4)
print("\n=== MUV10 matched factors — top 15 genes ===")
for f in [3, 6, 8]:
    print(f"MUV10_f{f}:", signatures[f"MUV10_f{f}"][:15])

# Are MUV10_f3 and MUV10_f6 the SAME (split) cell-cycle program?
cc_overlap = len(set(signatures["MUV10_f3"]) & set(signatures["MUV10_f6"]))
print(f"\nMUV10_f3 ∩ MUV10_f6 (both cell-cycle?): {cc_overlap}/30 shared genes")

# Same question for the other tumors that split cell-cycle:
print("\n=== split cell-cycle check (top-30 ∩ across the tumor's CC factors) ===")
for t, pair in [("BCH869", (7, 8)), ("BCH1126", (2, 4)), ("MUV10", (3, 6))]:
    a, b = pair
    ov = len(set(signatures[f"{t}_f{a}"]) & set(signatures[f"{t}_f{b}"]))
    print(f"  {t}: f{a} ∩ f{b} = {ov}/30")

# How many MUV10 unmatched factors are dominated by pseudogenes/LOC/antisense?
import re
def junk_frac(gene_list):
    junk = [g for g in gene_list if re.search(r"^(LOC\d|LINC|MIR\d)|-AS\d?$|P\d+$|orf", g)
            or g.endswith("P")]
    return len(junk) / len(gene_list)

print("\n=== fraction of likely non-coding/pseudogene symbols, MUV10 factors ===")
for f in range(10):
    jf = junk_frac(signatures[f"MUV10_f{f}"])
    tag = "  <-- junk-heavy" if jf >= 0.3 else ""
    print(f"  MUV10_f{f}: {jf:.0%}{tag}")

=== MUV10 unmatched factors — top 15 genes ===
MUV10_f0: ['HERC2P4', 'LOC100190986', 'UGDH-AS1', 'OPHN1', 'LOC643406', 'ARHGEF26-AS1', 'LPAL2', 'ABCC9', 'LOC646214', 'TMEM212', 'LOC286437', 'SHISA9', 'METTL21A', 'ODF2L', 'ZBTB16']
MUV10_f1: ['PSMD6', 'ESD', 'PPA2', 'TRAPPC3', 'EMC2', 'TERF2IP', 'TMEM170A', 'KAT7', 'EFHA1', 'MRPS35', 'CD46', 'MRPL3', 'SNX6', 'NDUFA5', 'MIEN1']
MUV10_f2: ['FLAD1', 'LSM10', 'SIVA1', 'PSMB1', 'EIF3M', 'EXOSC5', 'TIMM17A', 'TCTN3', 'NUTF2', 'PDCD6', 'PRKRA', 'NTHL1', 'VPS45', 'NFU1', 'DCTPP1']
MUV10_f4: ['GTF3A', 'TAF12', 'TWF1', 'EEF1E1', 'PIGH', 'GGCT', 'PIGF', 'KCNIP1', 'RNFT1', 'SEC23B', 'PLAT', 'NOP58', 'BAD', 'FH', 'HIBCH']
MUV10_f5: ['RBM4B', 'RAB5A', 'ISG15', 'IFI6', 'TM2D3', 'ORMDL1', 'NDUFS4', 'ANXA7', 'RSL24D1', 'ATP6V1G1', 'LOC401397', 'PROSC', 'ALDH3A2', 'ACAT2', 'FKBP3']
MUV10_f7: ['MORN2', 'DCTN6', 'SFRP2', 'CTSK', 'BCS1L', 'TM4SF1', 'BLOC1S1', 'NDUFB3', 'SLC40A1', 'MRPS22', 'RTFDC1', 'SERPINA3', 'MRPS6', 'ACTR10', 'SGK1']
MUV10_f9: ['VIM', '

In [9]:
# ============================================================
# Step 7 — Cell 4: FREEZE  →  Filbin_step7.h5ad
# Stores: 60 signatures, per-tumor W (gene×factor), per-cell H
# activations (obsm, NaN-padded), full NNMF config + anchors.
# Round-trip verified.
# ============================================================
import numpy as np
import pandas as pd
import scanpy as sc

TUMORS = ["MUV1", "MUV5", "MUV10", "BCH836", "BCH869", "BCH1126"]
K = 10

# ---- 1. Per-cell H activation block: (2456, 10), NaN where N/A ----
# Each malignant cell gets its OWN tumor's 10 factor activations.
# Non-malignant cells -> all NaN. Column j = "factor j of this cell's tumor".
H_block = np.full((adata.n_obs, K), np.nan, dtype=np.float64)
obs_pos = {name: i for i, name in enumerate(adata.obs_names)}
for t in TUMORS:
    names = cells_by_tumor[t]                 # row order of H_by_tumor[t]
    rows = [obs_pos[n] for n in names]
    H_block[rows, :] = H_by_tumor[t]
adata.obsm["X_nmf_H"] = H_block
# which tumor each cell's H columns belong to (for unambiguous reading)
adata.obs["nmf_tumor"] = adata.obs["patient"].astype(str)
adata.obs.loc[~adata.obs["is_malignant"], "nmf_tumor"] = "NA_nonmalignant"

# ---- 2. W stack: (6 tumors × 8494 genes × 10 factors) in uns ----
# Stored as a dict of 2D arrays (h5ad-safe), one per tumor.
W_store = {t: W_by_tumor[t].astype(np.float32) for t in TUMORS}

# ---- 3. Signatures: 60 lists, stored as a padded 60×30 symbol array ----
sig_names = [f"{t}_f{f}" for t in TUMORS for f in range(K)]
sig_matrix = np.array([signatures[s] for s in sig_names], dtype=object)  # (60, 30)

# ---- 4. Concordance table (the validation evidence) ----
concordance = overlap[prog_names + ["best_match", "best_overlap"]].copy()

# ---- 5. Provenance / anchors ----
adata.uns["step7"] = {
    "method": "per-tumor NNMF, k=10, on centered_within_tumor (Er) with negatives "
              "clipped to 0 (Filbin non-negativity step)",
    "substrate_layer": "centered_within_tumor",
    "nonneg_rule": "np.clip(Er, 0, None) per tumor on malignant cells only",
    "sklearn_NMF_config": {
        "n_components": K, "init": "random", "solver": "mu",
        "beta_loss": "frobenius", "max_iter": 1000, "random_state": 42,
    },
    "rationale_config": "Matlab nnmf analogue: multiplicative-update + random init + "
                        "frobenius loss; one run per tumor (single seed), per Filbin.",
    "n_factors_total": len(sig_names),          # 60
    "top_n_genes_per_factor": 30,
    "factor_loading_source": "W (model.components_.T), full 8494-gene space",
    "tumors": TUMORS,
    "n_malignant_per_tumor": {t: int(len(cells_by_tumor[t])) for t in TUMORS},
    "reconstruction_err": {t: recon_err[t] for t in TUMORS},
    "signature_names": sig_names,               # row order of sig_matrix
    "signatures": sig_matrix,                    # (60, 30) gene symbols
    "concordance_vs_tableS5": concordance.reset_index().to_dict(orient="list"),
    "tableS5_program_sizes": {k: len(v) for k, v in prog_lists.items()},
    "findings": (
        "5/6 tumors recover 6-8 of Filbin's 5 programs; cell-cycle universal (18-27 "
        "overlap, all 6). Cell-cycle resolves into G1/S + G2/M sub-phases in tumors where "
        "it splits (f-pairs share only ~4/30; biological phases, not redundancy). "
        "Lineage programs recur with per-tumor variation (MUV5 complete; BCH869 AC-dominant; "
        "BCH1126 OC-dominant). MUV10 is cycling/OPC-leaning, low OC/AC differentiation; its "
        "non-lineage factors are translational/IFN/stress (f1/f2/f5/f9) + one junk factor "
        "(f0, 30% pseudogene/LOC). Non-lineage recurrent programs (stress, translation) are "
        "expected and are NOT forced into Filbin labels."
    ),
    "deviations": "D10 (NNMF config: sklearn mu/random/frobenius as Matlab nnmf analogue, "
                  "single seed 42 — result is seed-dependent by construction, logged).",
}
adata.uns["step7_W"] = W_store

# ---- 6. Write + round-trip ----
OUT = "../data/processed/Filbin_step7.h5ad"
adata.write_h5ad(OUT, compression="gzip")
print(f"wrote {OUT}")

chk = sc.read_h5ad(OUT)
print("\nRound-trip:")
print(f"  shape:            {chk.shape}")                       # (2456, 8494)
print(f"  obsm X_nmf_H:     {chk.obsm['X_nmf_H'].shape}")       # (2456, 10)
print(f"  uns step7:        {'step7' in chk.uns}")
print(f"  uns step7_W keys: {sorted(chk.uns['step7_W'].keys())}")
print(f"  n signatures:     {len(chk.uns['step7']['signature_names'])}")
print(f"  sig_matrix shape: {np.array(chk.uns['step7']['signatures']).shape}")

# H sanity: malignant cells non-NaN, NM cells NaN
mal = chk.obs["is_malignant"].to_numpy()
H = chk.obsm["X_nmf_H"]
assert not np.isnan(H[mal]).any(),  "malignant cell has NaN activation"
assert np.isnan(H[~mal]).all(),     "non-malignant cell has non-NaN activation"
# anchors
assert chk.shape == (2456, 8494)
assert len(chk.uns["step7"]["signature_names"]) == 60
assert chk.uns["step7"]["sklearn_NMF_config"]["solver"] == "mu"
assert all(chk.uns["step7"]["n_malignant_per_tumor"][t] == n
           for t, n in {"MUV5":704,"BCH869":489,"BCH836":441,
                        "MUV10":286,"BCH1126":263,"MUV1":146}.items())
print("\nFilbin_step7.h5ad round-trips cleanly; anchors hold.")

wrote ../data/processed/Filbin_step7.h5ad

Round-trip:
  shape:            (2456, 8494)
  obsm X_nmf_H:     (2456, 10)
  uns step7:        True
  uns step7_W keys: ['BCH1126', 'BCH836', 'BCH869', 'MUV1', 'MUV10', 'MUV5']
  n signatures:     60
  sig_matrix shape: (60, 30)

Filbin_step7.h5ad round-trips cleanly; anchors hold.
